## Cleaning up a messy dataset

Source:

Ibrahim Salami: I Cleaned a Messy CSV File Using Pandas. towards data science, 2025-11-26.

https://towardsdatascience.com/i-cleaned-a-messy-csv-file-using-pandas-heres-the-exact-process-i-follow-every-time/


![](img/clean-up.jpg)

---

### The dataset

In this notebook, we are going to use a dataset from [kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows?resource=download), namely the **Netflix_Movies_and_TV_Shows** dataset.

Netflix is one of the most popular media and video streaming platforms. They have over 8000 movies or tv shows available on their platform, as of mid-2021, they have over 200M Subscribers globally. This tabular dataset consists of listings of all the movies and tv shows available on Netflix, along with details such as cast, directors, ratings, release year, duration, etc.

---

### Overview: Standard cleaning workflow


The standard cleaning workflow is adapted from:

Ibrahim Salami: I Cleaned a Messy CSV File Using Pandas. towards data science, 2025-11-26.

https://towardsdatascience.com/i-cleaned-a-messy-csv-file-using-pandas-heres-the-exact-process-i-follow-every-time/



The standard cleaning workflow consists of 5 simple stages.

1.    Load
1.    Inspect
1.    Clean
1.    Value Standardization
1.    Export

In [37]:
import pandas as pd
pd.options.display.max_rows = 100
pd.options.display.width = 800

---

#### Load

There are some things to keep in mind before loading your dataset. However, this is an optional step, and we probably wouldn’t encounter most of these issues in our dataset. But it doesn’t hurt to know these things. Here are some key things to consider while loading.

**Encoding issues** (utf-8, latin-1): 
Encoding defines how characters are stored as bytes in the file. Python and Pandas usually default to **UTF-8**, which handles most modern text and special characters globally. However, if the file was created in an older system or a non-English environment, it might use a different encoding, most commonly **Latin-1**.

So if you try to read a Latin-1 file with UTF-8, Pandas will encounter bytes it doesn’t recognise as valid UTF-8 sequences. You’ll typically see a `UnicodeDecodeError` when you try to read a CSV with encoding issues.

If perhaps the default load fails, you could try to specify a different encoding:

In [ ]:
# First attempt (the default)
try:
    df = pd.read_csv('datasets/Netflix Streaming Data/Netflix Streaming Data.csv')
except UnicodeDecodeError:
    print("UnicodeDecodeError encountered. Trying a different encoding ...")
# Second attempt with a common alternative
df = pd.read_csv('datasets/Netflix Streaming Data/Netflix Streaming Data.csv', encoding='latin-1')

**Wrong delimiters**: 
CSV stands for “Comma Separated Values,” but in reality, many files use other characters as separators, like semicolons (common in Europe), tabs, or even pipes (|). Pandas typically defaults to the comma (,).

So, if your file uses a semicolon (;) but you load it with the default comma delimiter, Pandas will treat the entire row as a single column. The result would be a DataFrame with a single column containing entire lines of data, making it impossible to work with.

The fix is pretty simple. You can try checking the raw file (opening it in a text editor like VS Code or Notepad++ is best) to see what character separates the values. Then, pass that character to the sep argument like so

In [ ]:
# If the file uses semicolons
df = pd.read_csv('datasets/Netflix Streaming Data/Netflix Streaming Data.csv', sep=';')

# If the file uses tabs (TSV)
df = pd.read_csv('datasets/Netflix Streaming Data/Netflix Streaming Data.csv', sep='\t')

**Columns that import incorrectly**: 
Sometimes, Pandas guesses the data type for a column based on the first few rows, but later rows contain unexpected data (e.g., text mixed into a column that started with numbers).

For instance, Pandas may correctly identify 0.1, 0.2, 0.3 as floats, but if row 100 contains the value N/A, Pandas might force the entire column into an object (string) type to accommodate the mixed values. This sucks because you lose the ability to perform fast, vectorised numeric operations on that column until you clean up the bad values.

To fix this, I use the dtype argument to tell Pandas what data type a column should be explicitly. This prevents silent type casting.

In [38]:
df = pd.read_csv('datasets/Netflix Streaming Data/Netflix Streaming Data.csv', dtype={'release_year': int})

**Reading the first few rows**: 
You could save time by checking the first few rows directly during the loading process using the nrows parameter. This is great, especially when you’re working with large datasets, as it allows you to test encoding and delimiters without loading the entire 10 GB file.

In [ ]:
# Load only the first 50 rows to confirm encoding and delimiter
print(df.head(50))

Once you’ve confirmed the arguments are correct, you can load the full file.

Let’s load the Employee dataset. We don’t expect to see any issues here.

In [55]:
df = pd.read_csv('datasets/Netflix Streaming Data/Netflix Streaming Data.csv')
df.shape

(8807, 12)

---

#### Inspect

**Understanding the Boundaries**:
I always start with a visual check. I use `df.head()` and `df.tail()` to see the first and last five rows. This is a quick sanity check to see if all columns look aligned and if the data visually makes sense.

In [40]:
print (df.head())
print ('\n')
print (df.tail())

  show_id     type                  title         director                                               cast        country          date_added  release_year rating   duration                                          listed_in                                        description
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson                                                NaN  United States  September 25, 2021          2020  PG-13     90 min                                      Documentaries  As her father nears the end of his life, filmm...
1      s2  TV Show          Blood & Water              NaN  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa  September 24, 2021          2021  TV-MA  2 Seasons    International TV Shows, TV Dramas, TV Mysteries  After crossing paths at a party, a Cape Town t...
2      s3  TV Show              Ganglands  Julien Leclercq  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...            NaN  September 24, 2021          2021  TV-

**Spotting Datatype Problems and Missingness**: 
This is the most critical method. It tells me the column names, the data types (Dtype), and the exact number of non-null values.


In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


Pandas has special ways of dealing with missing data depending on the data type. As you may have already noticed, certain fields in a CSV file show up as `NaN` (Not a Number) in a Pandas DataFrame. 

To filter and count the number of missing/not missing values in a dataset, we can use the special `.isna()` and `.notna()` methods on a DataFrame or Series object.

Kepp in mind that the `.count()` method always excludes NaN values, so we can count the number of available values in each column. The total number of rows in each column (`size`) can be used to find the percentage of not blank data in every column.

The `.isna()` and `.notna()` methods return True/False pairs for each row, which we can use to filter the DataFrame for any rows that have information in a given column. 

In [42]:
print(df['director'].count())
print(df['director'].size)
print(df['director'].notna().sum())
print(df['director'].isna().sum())
print(df['director'].notna().sum()/df['director'].size)

6173
8807
6173
2634
0.7009197229476553


**Data Integrity Check: Duplicates and Unique Counts**: 

- Checking for duplicate rows.

In [43]:
df.duplicated().sum()

np.int64(0)

This is perfect! It means we don’t have identical rows cluttering up our dataset.

- Checking Unique Values (`df.nunique()`): Count number of distinct elements per column (default). I use this to understand the diversity within each column. Low counts in categorical columns are fine, but I look for columns that should be unique but aren’t, or columns that have too many unique values, suggesting typos.

In [56]:
df.nunique()

show_id         8807
type               2
title           8807
director        4528
cast            7692
country          748
date_added      1767
release_year      74
rating            17
duration         220
listed_in        514
description     8775
dtype: int64

-
    - `show_id` has 8807 unique values. This is perfect.
    - `type` has 2 unique values, namely _TV Show_ and _Movie_. This is perfect.
    - `title` has 8807 unique values. This is perfect.
    - `director` has 4528 unique values. To my opinion, this is a really large number.
    - `cast` has 7692 unique values. That needs inspection.
    - `country` has 748 unique values. This large number may result from the fact that we have different combinations of countries in this column.
    - `date_added` has 1767 unique values. Ok.
    - `release_year` has 74 unique values. We may conclude that Netflix also has rather old movies.
    - `rating` has 17 unique values. That needs inspection.
    - `duration` has 220 unique values. This is a funny column.
    - `listed_in` has 514 unique values. Very high number for a categorical column. That needs inspection.
    - `description` has only 8775 unique values. Crazy.


**Catching Odd and Impossible Values**: 
I use `df.describe()` to get a statistical summary of all my numerical columns. This is the place where truly impossible values—the “red flags”—show up instantly. I mostly focus on the min and max rows.

In [45]:
df.describe()

,release_year
count,8807.000000
mean,2014.180198
std,8.819312
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


---

#### Clean

**The Consistency Rule—Standardising Column Names and Setting the Index**:
Before I do any serious data manipulation, I enforce strict consistency on column names. Why? Because typing df['Show ID '] accidentally instead of df['show_id'] is a silent, frustrating error. Once the names are clean, I set the index.

My golden rule is snake_case and lowercase everywhere, and ID columns should be the index.

I use a simple command to strip whitespace, replace spaces with underscores, and convert everything to lowercase.

In [ ]:
# The Standardization Command
df.columns = df.columns.str.lower().str.replace(' ', '_').str.strip()
df.columns

Optional: We might move on to set `show_id` as an index.

In [ ]:
# This is crucial for efficient lookups and clean merges later.
df.set_index('show_id', inplace=True)

# Let’s review it real quick
df.head()

We revert the last step.

In [ ]:
df = df.reset_index()
df.head()

**Handling Missing Values**: 
Finally, we address the gaps revealed by `df.info()`:

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


**Missing `cast` values**: We remove rows with missing `cast` values. Thus, we get rid of 825 rows.

In [57]:
# Removal: Drop rows missing values for 'cast'
df = df.dropna(subset=['cast'])
df = df.reset_index(drop=True)
print(f"Rows after dropping missing cast values: {len(df)}")
df.info()

Rows after dropping missing cast values: 7982
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7982 entries, 0 to 7981
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       7982 non-null   object
 1   type          7982 non-null   object
 2   title         7982 non-null   object
 3   director      5700 non-null   object
 4   cast          7982 non-null   object
 5   country       7305 non-null   object
 6   date_added    7972 non-null   object
 7   release_year  7982 non-null   int64 
 8   rating        7978 non-null   object
 9   duration      7979 non-null   object
 10  listed_in     7982 non-null   object
 11  description   7982 non-null   object
dtypes: int64(1), object(11)
memory usage: 748.4+ KB


---

#### Value Standardization

The DataFrame now has the right structure, but the values inside are still dirty. This step is about consistency. If “IT,” “i.t,” and “Info. Tech” all mean the same department, we need to force them into a single, clean value (“IT”). This prevents errors in grouping, filtering, and any statistical analysis based on categories.

**Rating Values—What a mess!** 
Let's create a set of all unique rating values to see what ratings are present in the dataset.


In [51]:
# Create a set of unique rating values
rating_set = set(df['rating'].unique())
print(f"Unique ratings ({len(rating_set)}):")
print(rating_set)


Unique ratings (18):
{'84 min', '74 min', 'PG-13', nan, 'TV-Y7-FV', 'TV-MA', 'TV-Y', 'TV-Y7', 'NC-17', 'R', 'G', 'UR', 'TV-14', '66 min', 'PG', 'TV-PG', 'NR', 'TV-G'}


In [52]:
row = df[df['rating'] == '66 min']
print(row)

     show_id   type                                 title    director        cast        country       date_added  release_year  rating duration listed_in                                        description
5289   s5814  Movie  Louis C.K.: Live at the Comedy Store  Louis C.K.  Louis C.K.  United States  August 15, 2016          2015  66 min      NaN    Movies  The comic puts his trademark hilarious/thought...


There certainly is a problem with the `rating` column, as the example shows: `rating` and `duration` seem to be mixed up. We don't care for a solution here, as we are not interested in the ratings for the moment.

**Converting Date Columns—The date_added Fix**: 
The `date_added` column is usually read in as a string (object) type, which makes time-series analysis impossible. This means we have to convert it to a proper Pandas datetime object. 
`pd.to_datetime()` is the core function. we use `errors='coerce'` as a safety net; if Pandas can’t parse a date, it converts that value to NaT (Not a Time), which is a clean null value, preventing the whole operation from crashing.

In [ ]:
print(df['date_added'].head())
print(f"\nOriginal dtype: {df['date_added'].dtype}")

# Convert the date_added column to datetime objects
df['date_added'] = pd.to_datetime(df['date_added'], format='%B %d, %Y', errors='coerce')

print(f"Converted dtype: {df['date_added'].dtype}")
df['date_added'].head()

0    September 24, 2021
1    September 24, 2021
2    September 24, 2021
3    September 24, 2021
4    September 24, 2021
Name: date_added, dtype: object

Original dtype: object
Converted dtype: datetime64[ns]


0   2021-09-24
1   2021-09-24
2   2021-09-24
3   2021-09-24
4   2021-09-24
Name: date_added, dtype: datetime64[ns]

: 

Following format codes apply:

- %B = Full month name (September)
- %b = Abbreviated month name (Sep)
- %d = Day of month (01-31)
- %Y = 4-digit year (2001)
- %y = 2-digit year (01)
- %m = Month as number (09)

---

#### Export

Before closing the notebook, I always perform one last audit to ensure everything is perfect, and then I export the data so I can perform analysis on it later.

**The Final Data Quality Check**: 
This is quick. I re-run the two most critical inspection methods to confirm that all my cleaning commands actually worked:

- `df.info()`: I confirm there are no more missing values in the critical columns (age, salary) and that the data types are correct (phone is a string, join_date is datetime).
- `df.describe()`: I ensure the statistical summary shows plausible numbers. The Phone column should now be absent from this output (since it’s a string), and Age and Salary should have logical minimum and maximum values.
If these checks pass, I know the data is reliable.

**Exporting the Clean Dataset**: 
The final step is to save this cleaned version of the data. I usually save it as a new CSV file to keep the original messy file intact for reference. I use index=False here if I don’t want the employee_id (which is now the index) to be saved as a separate column, or index=True if I want to save the index as the first column in the new CSV.

In [ ]:
# Exporting the clean DataFrame to a new CSV file
# We use index=True to keep our primary key (employee_id) in the exported file
df.to_csv('datasets/data/cleaned_employee_data.csv', index=True)

By exporting with a clear, new filename (e.g., _clean.csv), you officially mark the end of the cleaning phase and provide a clean slate for the next phase of the project.

---

### Overview: Application-specific cleaning workflow